# core

> database, llm, and run helpers

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
#| default_exp core

## Database

In [ ]:
#| export
from fastcore.all import *
from fastlite import *

In [ ]:
??Queryable.schema

In [ ]:
#| export
@patch(as_prop=True)
def schema(self:Queryable) -> str:
    "SQL schema for this table or view."
    return hl_md(self.db.execute(
        "select sql from sqlite_master where name = ?", (self.name,)
    ).fetchone()[0], lang='sql')

In [ ]:
#| export
db = database(f'db.db')

In [ ]:
db

In [ ]:
#| export
class Video: id:int; title:str; overview:str; transcript:str; pdfs:str; length:int; sample_rate:int
class Frame: id:int; video_id:int; frame_number:int
class Run: id:int; deploy_time:str; finish_time:str; request_timer:str; video_id:int; model:str; usage:str; num_frames:int; description:str
class RunFrame: run_id:int; frame_id:int; type:str; system_prompt:str; prompt:str; description:str; usage:str

In [ ]:
#| export
videos = db.create(Video, transform=True)
frames = db.create(Frame, transform=True, foreign_keys=[('video_id', 'video', 'id')])
runs = db.create(Run, transform=True)
runframes = db.create(RunFrame, pk=['run_id', 'frame_id', 'type'], foreign_keys=[('run_id', 'run', 'id'), ('frame_id', 'frame', 'id')], transform=True)

In [ ]:
videos, frames, runs, runframes

In [ ]:
#| export
from operator import methodcaller
delete_where = methodcaller('delete_where')
def reset_tables(): list(map(delete_where, [videos,frames,runs,runframes]))

In [ ]:
pth = L(Path('../../data/frames').glob('frames_*'))
pth.sort(key=lambda x: int(x.stem.split('_')[-1]))

In [ ]:
pth[:5]

## Scraping

In [ ]:
#| export
import os, regex as re
from html2text import HTML2Text as H2T

In [ ]:
#| export
hdrs = {'Authorization': f'Bearer {os.environ["JINA"]}', 'X-Return-Format': 'html'}
def url2html(url): return urlread(f'https://r.jina.ai/{url}', headers=hdrs)

In [ ]:
#| export
def fetch_overview(html): return re.sub(r'<[^>]+>', '', re.findall(r'<div id="tabs-1">\s*(.*?)\s*</div>', html, re.DOTALL)[0])

In [ ]:
#| export
def fetch_ts(html, pattern=r'<table class="sticky.*?</table>'): return H2T().handle(re.search(pattern, html, re.DOTALL).group(0))

In [ ]:
#| export
def fetch_att(html, pattern=r'>([^<]+\.pdf)<'): return re.findall(pattern, html)

## LLM Helpers

In [ ]:
#| export
from cachy import enable_cachy, disable_cachy

In [ ]:
enable_cachy()

In [ ]:
#| export
from fastllm.types import Msg, Part, PartType, Completion

In [ ]:
?Msg

In [ ]:
?Part

In [ ]:
#| export
def user(
    txt,
    img=None # accepts either a URL or a base64 data URL
):
    if img is None: return Msg(role='user', content=[Part(type=PartType.text, text=txt)])
    else: return Msg(role='user', content=[Part(type=PartType.input_image, text=img), Part(type=PartType.text, text=txt)])

In [ ]:
user('你好')

In [ ]:
#| export
def assistant(txt, data={'citations':[]}): return Msg(role='assistant', content=[Part(type=PartType.text, text=txt, data=data)])

In [ ]:
#| export
from fastllm.acomplete import acomplete

In [ ]:
?acomplete

In [ ]:
#| export
async def stream(msgs:list=None, model:str='', max_think=float('inf'), usage:bool=True, display:bool=True, **kwargs):
    'Stream a response, printing text/thinking as it arrives. Returns the final completion.'
    assert msgs is not None, 'no messages provided'
    assert model!='', 'no model name provided'
    think_cnt, seen_txt = 0, False
    async for o in await acomplete(msgs, model, stream=True, **kwargs):
        if not isinstance(o, Completion) and display:
            if o.get('thinking') and think_cnt<max_think: print('🤔', end='', flush=True)
            if txt:=o.get('text'): print(f"{'\n\n' if not seen_txt else ''}{txt}", end='', flush=True) or not seen_txt and (seen_txt:=True)
            think_cnt+=1
    if display: print()
    return o

In [ ]:
#| export
@delegates(stream, keep=True)
def session(**kwargs): return partial(stream, **kwargs)

In [ ]:
?session

### Dialog

In [ ]:
#| export
class Dialog:
    def __init__(self, msgs=None, context=''): 
        if msgs is None: msgs = []
        store_attr('msgs, context', self)

### Image

In [ ]:
#| export
from base64 import b64encode
def img2b64(path): return 'data:image/png;base64,'+b64encode(Path.read_bytes(path)).decode()

## Models

In [ ]:
#| export
models = AttrDict(
    gemma26b       =AttrDict(name='google/gemma-4-26b-a4b-it:free', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    gemma31b       =AttrDict(name='google/gemma-4-31b-it', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    grok4p3        =AttrDict(name='x-ai/grok-4.3', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    grok4p5        =AttrDict(name='x-ai/grok-4.5', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    qwen3p7plus    =AttrDict(name='qwen/qwen3.7-plus', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    qwen3p7max     =AttrDict(name='qwen/qwen3.7-max', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    mistral3p5     =AttrDict(name='mistralai/mistral-medium-3-5', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    mistral4       =AttrDict(name='mistralai/mistral-small-2603', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    n2mini         =AttrDict(name='nex-agi/nex-n2-mini', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    n2pro          =AttrDict(name='nex-agi/nex-n2-pro', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    hy3            =AttrDict(name='tencent/hy3', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    fugu           =AttrDict(name='sakana/fugu-ultra', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    glm4p6v        =AttrDict(name='z-ai/glm-4.6v', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    glm5p2         =AttrDict(name='z-ai/glm-5.2', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    kimi2p6        =AttrDict(name='moonshotai/kimi-k2.6', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    kimi2p7code    =AttrDict(name='moonshotai/kimi-k2.7-code', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    kimi3    =AttrDict(name='moonshotai/kimi-k3', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    nemotron_nano  =AttrDict(name='nvidia/nemotron-3-nano-30b-a3b', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    nemotron_omni  =AttrDict(name='nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    nemotron_ultra =AttrDict(name='nvidia/nemotron-3-ultra-550b-a55b', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    minimax_m3     =AttrDict(name='minimax/minimax-m3', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    step3p7        =AttrDict(name='stepfun/step-3.7-flash', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    mimo_v2p5      =AttrDict(name='xiaomi/mimo-v2.5', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    mimo_v2p5pro   =AttrDict(name='xiaomi/mimo-v2.5-pro', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    seed2p0mini    =AttrDict(name='bytedance-seed/seed-2.0-mini', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
    seed2p0lite    =AttrDict(name='bytedance-seed/seed-2.0-lite', kw=AttrDict(vendor_name='openrouter', reasoning_effort='high')),
)

## Prompts

In [ ]:
#| export
caveman_prompt = """
Respond terse like smart caveman. All technical substance stay. Only fluff die.

## Persistence

ACTIVE EVERY RESPONSE. No revert after many turns. No filler drift. Still active if unsure.

## Rules

Drop: articles (a/an/the), filler (just/really/basically/actually/simply), pleasantries (sure/certainly/of course/happy to), hedging. Fragments OK. Short synonyms (big not extensive, fix not "implement a solution for"). Technical terms exact. Code blocks unchanged. Errors quoted exact.

Pattern: `[thing] [action] [reason]. [next step].`

Not: "Sure! I'd be happy to help you with that. The issue you're experiencing is likely caused by..."
Yes: "Bug in auth middleware. Token expiry check use `<` not `<=`. Fix:"

## Intensity

Example — "Why React component re-render?"
- "New object ref each render. Inline object prop = new ref = re-render. Wrap in `useMemo`."

Example — "Explain database connection pooling."
- "Pool reuse open DB connections. No new connection per request. Skip handshake overhead."

## Auto-Clarity

Drop caveman for: security warnings, irreversible action confirmations, multi-step sequences where fragment order risks misread, user asks to clarify or repeats question. Resume caveman after clear part done.

Example — destructive op:
> **Warning:** This will permanently delete all rows in the `users` table and cannot be undone.
> ```sql
> DROP TABLE users;
> ```
> Caveman resume. Verify backup exist first.
"""

In [ ]:
#| export
prompt_template = '{} Report only what is directly observable — no speculation, no assumptions, no decorative language, and no observations beyond what was specifically requested.'
prompts = AttrDict(
    environment=prompt_template.format('Detail the environment presented in the given frame.'),
    blackboard=prompt_template.format('Detail the contents present on chalkboard/blackboard/whiteboard presented in the given frame.'),
    teacher=prompt_template.format("Detail the teacher presented in the given frame, including but not limited to, the teacher's expressions, emotions, gestures, and interactions/dynamics if any."),
    students=prompt_template.format("Detail the students presented in the given frame, including but not limited to, the students' expressions, emotions, gestures, and interactions/dynamics between each other if any."),
)

In [ ]:
#| export
sys_prompt = f"""
RESPONSE STYLE
==============
{caveman_prompt}

ROLE
====
You are a classroom observation analyst. You receive a time-ordered window of per-frame descriptions, each broken down by category (ENVIRONMENT, BLACKBOARD, TEACHER, STUDENTS) at 2-second intervals.

Synthesize these fragments into a single coherent narrative of what is happening across the window, captured by the camera. Track continuity and change: who is doing what, how the classroom state evolves, and what the instructional activity is. Weave the categories together rather than listing them separately. Do not use bullet points. Write in flowing paragraphs.

Report only what is directly supported by the descriptions. No speculation, no assumptions, no decorative language. If something cannot be determined from the provided text, omit it.
"""

## Run Helpers

In [ ]:
#| export
from datetime import datetime
from zoneinfo import ZoneInfo
tz = ZoneInfo("Asia/Hong_Kong")

In [ ]:
#| export
from operator import attrgetter, itemgetter
def view_col(table, col, where=None, where_args=None): 
    if where is None: return L(table()).map(attrgetter(col))
    else: return L(table.rows_where(where=where, where_args=where_args)).map(itemgetter(col))

In [ ]:
#| export
def compute_usage(run_id):
    usgs = L(runframes.rows_where('run_id=?', (run_id,))).map(lambda r: dict2obj(loads(r['usage'])))
    if not usgs: return None
    tot = dict2obj({k1: ({k2: 0 for k2 in v1} if isinstance(v1,dict) else False if isinstance(v1,bool) else 0) for k1,v1 in usgs[0].items()})
    for u in usgs:
        for k1,v1 in u.items():
            if isinstance(v1,dict):
                for k2,v2 in v1.items(): tot[k1][k2] += v2
            else: tot[k1] += v1
    return tot

In [ ]:
#| export
from sys import maxsize
from fastprogress.fastprogress import NBMasterBar as master_bar

In [ ]:
#| export
async def deploy_run(video_id, session, pth, start, stop, step=1, run_id=None, cache=False):
    if not cache: 
        disable_cachy()
        print('!! Cache disabled')
    else: print('!! Using cache')

    run = runs.insert(id=run_id, deploy_time=datetime.now(tz), video_id=video_id, model=session.keywords['model'], num_frames=(stop-start)//step)

    print(f'╭─ Run #{run.id} ═══════════════════════════════╮\n│ Model    {session.keywords["model"]}\n│ Start    {start}\n│ Stop     {stop}\n│ Step     {step}\n│ Frames   {(stop-start)//step}\n│ Cache    {cache}\n│ Time     {datetime.now(tz).strftime("%H:%M:%S")}\n╰──────────────────────────────────────────────╯')

    for p in (mb:=master_bar(pth[start:stop:step])):
        mb.main_bar.comment = f'frame {p.stem.split("_")[1]}'
        ins_row = partial(runframes.insert, run_id=run.id, frame_id=p.stem.split('_')[1], system_prompt=session.keywords.get('system'))
        for t in mb.progress(L(prompts.keys())):
            mb.child.comment = f'running {t} prompt'
            r = await session([user(prompts[t], img=img2b64(p))])
            ins_row(type=t, prompt=prompts[t], description=r.message.text, usage=r.usage.raw)

    if not cache:
        enable_cachy()
        print('!! Cache enabled')

    runs.update(id=run.id, finish_time=datetime.now(tz), request_timer=None, usage=compute_usage(run.id))
    run = runs[run.id]
    tot = dict2obj(loads(run.usage))
    elapsed = datetime.fromisoformat(run.finish_time) - datetime.fromisoformat(run.deploy_time)
    print(f'╭─ Run #{run.id} Complete ═════════════════════╮\n│ Finish   {datetime.fromisoformat(run.finish_time).strftime("%H:%M:%S")}\n│ Elapsed  {str(elapsed).split(".")[0]}\n│ Cost     ${tot.cost:.4f} (HKD {tot.cost*7.84:.2f})\n╰──────────────────────────────────────────────╯')
    return run.id

In [ ]:
#| export
async def summarize_window(run_id, start, stop, step=1, session=None, cache=False):
    'Summarize a window of frames from runframes. Returns summary text.'
    assert session is not None, 'no session provided'
    
    if not cache: disable_cachy()
    
    rows = L(runframes.rows_where(where='run_id=? AND frame_id>=? AND frame_id<?', where_args=(run_id, start, stop)))
    grouped = rows.groupby(lambda r: r['frame_id'])
    fids = sorted(grouped.keys())
    if step > 1: fids = fids[::step]
    
    window = ''
    for fid in fids:
        prefix = f'\n\nTimestamp ({fid}s)\n'
        window += prefix + len(prefix.strip())*'='
        for d in grouped[fid]: window += f"\n\n--\n\n{d['type'].upper()}\n\n{d['description']}"
    
    from tiktoken import encoding_for_model
    enc = encoding_for_model('gpt-4o')
    win_tokens = len(enc.encode(window))
    
    t0 = datetime.now(tz)
    print(f'╭─ Summary Run #{run_id} ═══════════════════════╮\n│ Frames   {start}–{stop} (step {step})\n│ Model    {session.keywords["model"]}\n│ Window   {len(window)} chars / {win_tokens} tokens\n│ Cache    {cache}\n│ Start    {t0.strftime("%H:%M:%S")}\n╰──────────────────────────────────────────────╯')
    
    r = await session([user(window)], system=sys_prompt)
    t1 = datetime.now(tz)
    elapsed = t1 - t0
    
    if not cache: enable_cachy()
    
    summary = r.message.text
    sum_tokens = len(enc.encode(summary))
    reduction = (1 - sum_tokens/win_tokens)*100 if win_tokens else 0
    cost = r.usage.raw.get('cost', 0)
    
    print(f'╭─ Summary Complete ═══════════════════════════╮\n│ Finish   {t1.strftime("%H:%M:%S")}\n│ Elapsed  {str(elapsed).split(".")[0]}\n│ Summary  {len(summary)} chars / {sum_tokens} tokens\n│ Reduced  {reduction:.1f}%\n│ Cost     ${cost:.4f} (HKD {cost*7.84:.2f})\n╰──────────────────────────────────────────────╯')
    return summary

## Export -

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()